In [1]:
!pip install ml_collections

In [4]:
import os
import sys
import shutil
import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from tqdm import tqdm, trange
from glob import glob




from ml_collections import ConfigDict

In [2]:
sys.path.append('..')

In [3]:
from CRT_utils.CRT.core.model import Model
# from CRT_utils.CRT.core.config import create_config, save_config
from CRT_utils.CRT.littlehelper import *

from CRT_utils.CRT.test import test
# from utils.evaluate_uncertainty import evaluate_uncertainty
from CRT_utils.CRT.core.config import create_config, save_config
from CRT_utils.CRT.core.dataset import COCODataset, COCODatasetWithID
from CRT_utils.CRT.core.model import Model
from CRT_utils.CRT.core.metrics import AccuracyLogger


In [45]:
with open('../datasets/COCO18_dset_for_CRT_training/train.json', 'rb') as file:
    train_metadata = json.load(file)

In [46]:
train_metadata.keys()

dict_keys(['info', 'licenses', 'images', 'annotations', 'categories'])

In [47]:
all_coco_filename = list(map(lambda x: x.split('/')[-1], glob('../datasets/COCO18_dset_for_CRT_training/coco18_for_crt/*')))

In [48]:
all_coco_filename[0]

'000000327573.jpg'

In [49]:
final_img_metadata_ls = []

for i in trange(len(train_metadata['images'])):
    image_metadata = train_metadata['images'][i]
    
    if image_metadata['file_name'] in all_coco_filename:
        final_img_metadata_ls.append(image_metadata)
        
train_metadata['images'] = final_img_metadata_ls

100%|█████████████████████████████████| 118287/118287 [00:31<00:00, 3789.92it/s]


In [50]:
len(train_metadata['images'])

51500

In [51]:
img_id_ls = list(map(lambda x: x['id'], train_metadata['images']))

filtered_annotation_ls = []


for i in trange(len(train_metadata['annotations'])):
    annotation_metadata = train_metadata['annotations'][i]
    
    if annotation_metadata['image_id'] in img_id_ls:
        filtered_annotation_ls.append(annotation_metadata)

train_metadata['annotations'] = filtered_annotation_ls





100%|███████████████████████████████| 1607459/1607459 [05:05<00:00, 5261.61it/s]


In [52]:
with open('../datasets/COCO18_dset_for_CRT_training/train_metadata.json', 'rb') as file:
    train_metadata1 = json.load(file)

In [53]:
category_id_ls = list(map(lambda x: x['id'], train_metadata1['categories']))

In [54]:
len(category_id_ls)

18

In [55]:
train_metadata['categories'] = train_metadata1['categories']

In [56]:
filtered_annotation_ls = []


for i in trange(len(train_metadata['annotations'])):
    annotation_metadata = train_metadata['annotations'][i]
    
    if annotation_metadata['category_id'] in category_id_ls:
        filtered_annotation_ls.append(annotation_metadata)

train_metadata['annotations'] = filtered_annotation_ls

100%|██████████████████████████████| 884246/884246 [00:00<00:00, 3722558.83it/s]


In [57]:
len(train_metadata['annotations'])

202592

In [58]:
train_metadata['annotations'][0]

{'segmentation': [[578.17,
   274.95,
   576.92,
   271.19,
   578.8,
   263.05,
   579.43,
   259.29,
   577.55,
   256.15,
   575.04,
   251.77,
   571.28,
   249.26,
   568.15,
   244.87,
   568.15,
   239.23,
   573.79,
   233.59,
   577.55,
   232.34,
   584.44,
   232.34,
   591.33,
   229.2,
   595.09,
   229.83,
   595.72,
   229.83,
   602.61,
   229.83,
   605.75,
   234.22,
   605.75,
   240.48,
   605.75,
   246.12,
   601.36,
   251.14,
   600.73,
   251.77,
   600.11,
   257.41,
   599.48,
   266.81,
   597.6,
   271.82,
   596.35,
   272.45,
   588.83,
   272.45,
   579.43,
   273.07]],
 'area': 1169.2116500000013,
 'iscrowd': 0,
 'image_id': 206784,
 'bbox': [568.15, 229.2, 37.6, 45.75],
 'category_id': 64,
 'id': 18912}

In [60]:
img_id_in_annotations = list(map(lambda x: x['image_id'], train_metadata['annotations']))

In [61]:
len(set(img_id_in_annotations))

51500

In [59]:
with open('../datasets/COCO18_dset_for_CRT_training/train_metadata.json', 'w') as file:
    json.dump(train_metadata, file)

In [62]:
with open('../datasets/COCO18_dset_for_CRT_training/val.json', 'rb') as file:
    val_metadata = json.load(file)

In [63]:
img_id_ls = list(map(lambda x: x['id'], val_metadata['images']))

filtered_annotation_ls = []


for i in trange(len(val_metadata['annotations'])):
    annotation_metadata = val_metadata['annotations'][i]
    
    if annotation_metadata['image_id'] in img_id_ls:
        filtered_annotation_ls.append(annotation_metadata)

val_metadata['annotations'] = filtered_annotation_ls

100%|██████████████████████████████████| 69582/69582 [00:00<00:00, 79616.02it/s]


In [64]:
len(val_metadata['annotations'])

69582

In [65]:
val_metadata['categories'] = train_metadata1['categories']

In [66]:
filtered_annotation_ls = []


for i in trange(len(val_metadata['annotations'])):
    annotation_metadata = val_metadata['annotations'][i]
    
    if annotation_metadata['category_id'] in category_id_ls:
        filtered_annotation_ls.append(annotation_metadata)

val_metadata['annotations'] = filtered_annotation_ls

100%|████████████████████████████████| 69582/69582 [00:00<00:00, 2116497.41it/s]


In [67]:
len(val_metadata['annotations'])

8879

In [68]:
img_id_in_annotations = list(map(lambda x: x['image_id'], val_metadata['annotations']))

In [69]:
len(img_id_in_annotations)

8879

In [71]:
img_id_in_annotations = list(set(img_id_in_annotations))

In [72]:
final_img_metadata_ls = []

for i in trange(len(val_metadata['images'])):
    image_metadata = val_metadata['images'][i]
    
    if image_metadata['id'] in img_id_in_annotations:
        final_img_metadata_ls.append(image_metadata)
        
val_metadata['images'] = final_img_metadata_ls

100%|████████████████████████████████████| 5000/5000 [00:00<00:00, 73777.92it/s]


In [73]:
len(val_metadata['images'])

2252

In [74]:
with open('../datasets/COCO18_dset_for_CRT_training/val_metadata.json', 'w') as file:
    json.dump(val_metadata, file)

In [24]:
train_metadata['annotations'][0]

{'segmentation': [[239.97,
   260.24,
   222.04,
   270.49,
   199.84,
   253.41,
   213.5,
   227.79,
   259.62,
   200.46,
   274.13,
   202.17,
   277.55,
   210.71,
   249.37,
   253.41,
   237.41,
   264.51,
   242.54,
   261.95,
   228.87,
   271.34]],
 'area': 2765.1486500000005,
 'iscrowd': 0,
 'image_id': 558840,
 'bbox': [199.84, 200.46, 77.71, 70.88],
 'category_id': 58,
 'id': 156}

In [7]:
images_idx2id_images      = {}
images_idx2id_annotations = {}


for i in trange(len(train_metadata['images'])):
    
    image_metadata = train_metadata['images'][i]
    
    

{'license': 3,
 'file_name': '000000391895.jpg',
 'coco_url': 'http://images.cocodataset.org/train2017/000000391895.jpg',
 'height': 360,
 'width': 640,
 'date_captured': '2013-11-14 11:18:45',
 'flickr_url': 'http://farm9.staticflickr.com/8186/8119368305_4e622c8349_z.jpg',
 'id': 391895}

In [9]:
len(train_metadata['images'])

118287

In [10]:
len(train_metadata['annotations'])

1607459

In [30]:
len(train_metadata['annotations'])

884246

In [16]:
train_metadata['annotations'][0]

{'segmentation': [[239.97,
   260.24,
   222.04,
   270.49,
   199.84,
   253.41,
   213.5,
   227.79,
   259.62,
   200.46,
   274.13,
   202.17,
   277.55,
   210.71,
   249.37,
   253.41,
   237.41,
   264.51,
   242.54,
   261.95,
   228.87,
   271.34]],
 'area': 2765.1486500000005,
 'iscrowd': 0,
 'image_id': 558840,
 'bbox': [199.84, 200.46, 77.71, 70.88],
 'category_id': 58,
 'id': 156}

In [18]:
train_metadata['images'][0]

{'license': 4,
 'file_name': '000000522418.jpg',
 'coco_url': 'http://images.cocodataset.org/train2017/000000522418.jpg',
 'height': 480,
 'width': 640,
 'date_captured': '2013-11-14 11:38:44',
 'flickr_url': 'http://farm1.staticflickr.com/1/127244861_ab0c0381e7_z.jpg',
 'id': 522418}

51500

51500

884246

51500

In [36]:
len(category_id_ls_in_annotations)

884246

In [37]:
len(set(category_id_ls_in_annotations))

172

In [38]:
img_id_cat_id_ls_in_annotation = list(map(lambda x: str(x['category_id']) + str(x['image_id']), train_metadata['annotations']))

588555

# User define Variables (Arguments)

In [13]:
config_dict = ConfigDict()


config_dict['config']                = None
config_dict['outdir']                = '../CRT_utils/CRT_weights_and_config/CRT_COCO_all'
config_dict['checkpoint']            = None      # '../CRT_utils/CRT_weights_and_config/checkpoint1.tar'

config_dict['annotations']           = '../datasets/COCO18_dset_for_CRT_training/train_metadata.json'
config_dict['imagedir']              = '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt'

config_dict['test_annotations']      = '../datasets/COCO18_dset_for_CRT_training/val_metadata.json'
config_dict['test_imagedir']         = '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt_test'
config_dict['test_frequency']        = None

config_dict['epochs']                = 30
config_dict['save_frequency']        = None
config_dict['print_batch_metrics']   = None

config_dict['batch_size']            = 32
config_dict['learning_rate']         = None
config_dict['imbalance_reweighting'] = None
config_dict['num_decoder_heads']     = None
config_dict['num_decoder_layers']    = None
config_dict['uncertainty_gate_type'] = None
config_dict['uncertainty_threshold'] = 0
config_dict['weighted_prediction']   = None

In [14]:
cfg = create_config(config_dict)

Could not save git state to config.


In [6]:
dataset = COCODataset(
    cfg.annotations, 
    cfg.imagedir, 
    image_size      = (224,224), 
    normalize_means = [0.485, 0.456, 0.406], 
    normalize_stds  = [0.229, 0.224, 0.225]
)

dataloader = DataLoader(
    dataset, 
    batch_size  = cfg.batch_size, 
    num_workers = 1, 
    shuffle     = True, 
    pin_memory  = True, 
    drop_last   = True
)


-------------------------------
Annotation Counts
-------------------------------
potted plant                585
tv                          228
bottle                     1267
chair                      1561
car                        4415
stop sign                   298
clock                       262
cup                         629
fork                         60
knife                       287
bowl                        692
toilet                      555
laptop                      145
mouse                        82
keyboard                     98
microwave                    73
oven                        291
sink                        625
Total                     12153
-------------------------------



In [9]:
NUM_CLASSES     = dataset.NUM_CLASSES
cfg.num_classes = NUM_CLASSES

os.makedirs(config_dict.outdir, exist_ok=True)

save_config(cfg, config_dict.outdir)

print(cfg)

annotations: ../datasets/COCO18_dset_for_CRT_training/train_metadata.json
batch_size: 32
checkpoint: ../CRT_utils/CRT_weights_and_config/checkpoint1.tar
imagedir: ../datasets/COCO18_dset_for_CRT_training/coco18_for_crt
imbalance_reweighting: false
learning_rate: 1.0e-05
num_classes: 18
num_decoder_heads: 8
num_decoder_layers: 6
test_annotations: ../datasets/COCO18_dset_for_CRT_training/val_metadata.json
test_imagedir: ../datasets/COCO18_dset_for_CRT_training/coco18_for_crt_test
uncertainty_gate_type: learned
uncertainty_threshold: 0
weighted_prediction: false



In [10]:
model = Model.from_config(cfg)

/Users/nguyentuan/anaconda3/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/nguyentuan/anaconda3/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet169_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet169_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/densenet169-b2777c0a.pth" to /Users/nguyentuan/.cache/torch/hub/checkpoints/densenet169-b2777c0a.pth


100%|██████████████████████████████████████| 54.7M/54.7M [00:01<00:00, 33.2MB/s]


In [11]:
assert(model.TARGET_IMAGE_SIZE == model.CONTEXT_IMAGE_SIZE == dataset.image_size), "Image size from the dataset is not compatible with the encoder."

In [19]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps"  # macbook uses metal performance shaders to GPU accelearation
    if torch.backends.mps.is_available()
    else "cpu"
)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)


if cfg.imbalance_reweighting:
    class_weights = torch.true_divide(dataset.relative_annotation_counts.max(), dataset.relative_annotation_counts)
    criterion     = nn.CrossEntropyLoss(weight= class_weights.to(device))
else:
    criterion = nn.CrossEntropyLoss()

if cfg.checkpoint is not None:
    
    print("Initializing from checkpoint {}".format(cfg.checkpoint))
    
    checkpoint = torch.load(cfg.checkpoint, map_location="cpu")
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    start_epoch = checkpoint['epoch'] + 1
else:
    print("No checkpoint was passed.")
    
    model.to(device)
    start_epoch = 1

# Tensorboard
writer = SummaryWriter(log_dir=os.path.join(config_dict.outdir, "runs/{date:%Y-%m-%d_%H%M}".format(date=datetime.datetime.now())))

context_images, target_images, bbox, labels = next(iter(dataloader))

writer.add_images("context_image_batch", context_images) # add example context image batch to tensorboard log
writer.add_images("target_image_batch", target_images)   # add example target image batch to tensorboard log

with warnings.catch_warnings(): # add_graph method is known to issue a warning
    warnings.simplefilter("ignore")
    writer.add_graph(model, input_to_model=[context_images.to(device), target_images.to(device), bbox.to(device)]) # add model graph to tensorboard log

accuracy_logger_main_branch = AccuracyLogger(dataset.idx2label)

No checkpoint was passed.


KeyError: Caught KeyError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/Users/nguyentuan/anaconda3/lib/python3.11/site-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/Users/nguyentuan/anaconda3/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/nguyentuan/anaconda3/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 52, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/Users/nguyentuan/Mirror/STUDY_DOCUMENTS/FYP/TCT code/TCT-visual-search/TCT notebooks/../CRT_utils/CRT/core/dataset.py", line 106, in __getitem__
    image = Image.open(self.id2file[annotation["ima

In [20]:
dataset.id2file

{29913: '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt/000000029913.jpg',
 332654: '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt/000000332654.jpg',
 208589: '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt/000000208589.jpg',
 334405: '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt/000000334405.jpg',
 158952: '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt/000000158952.jpg',
 308599: '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt/000000308599.jpg',
 351053: '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt/000000351053.jpg',
 157269: '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt/000000157269.jpg',
 358342: '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt/000000358342.jpg',
 8665: '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt/000000008665.jpg',
 322654: '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt/000000322654.jpg',
 110196: '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt/00

In [ ]:
with open('../datasets/COCO18_dset_for_CRT_training/train_metadata.json', 'rb') as file:
    train_metadata = json.load(file)

In [14]:
train_dict = {}

train_dict['images']      = []
train_dict['annotations'] = []
train_dict['categories']  = train_metadata['categories']
train_dict['licenses']    = train_metadata['licenses']

for i in tqdm(range(len(train_metadata['images']))):
    
    if train_metadata['annotations'][i]['category_id'] in all_categories:
        train_dict['images'].append(train_metadata['images'][i])
        train_dict['annotations'].append(train_metadata['annotations'][i])
    

100%|████████████████████████████████| 51500/51500 [00:00<00:00, 2892429.78it/s]


In [15]:
with open('../datasets/COCO18_dset_for_CRT_training/train_metadata.json', 'w') as file:
    json.dump(train_dict, file)

In [21]:
with open('../datasets/COCO18_dset_for_CRT_training/train_metadata.json', 'rb') as file:
    train_metadata = json.load(file)

In [22]:
train_metadata.keys()

dict_keys(['images', 'annotations', 'categories', 'licenses'])

In [23]:
train_metadata['annotations'][0]

{'segmentation': [[578.17,
   274.95,
   576.92,
   271.19,
   578.8,
   263.05,
   579.43,
   259.29,
   577.55,
   256.15,
   575.04,
   251.77,
   571.28,
   249.26,
   568.15,
   244.87,
   568.15,
   239.23,
   573.79,
   233.59,
   577.55,
   232.34,
   584.44,
   232.34,
   591.33,
   229.2,
   595.09,
   229.83,
   595.72,
   229.83,
   602.61,
   229.83,
   605.75,
   234.22,
   605.75,
   240.48,
   605.75,
   246.12,
   601.36,
   251.14,
   600.73,
   251.77,
   600.11,
   257.41,
   599.48,
   266.81,
   597.6,
   271.82,
   596.35,
   272.45,
   588.83,
   272.45,
   579.43,
   273.07]],
 'area': 1169.2116500000013,
 'iscrowd': 0,
 'image_id': 206784,
 'bbox': [568.15, 229.2, 37.6, 45.75],
 'category_id': 64,
 'id': 18912}

In [12]:
for metadata in train_metadata['annotations']:
    if metadata['category_id'] not in all_categories:
        pass

In [39]:
train_metadata['categories']

[{'id': 3, 'name': 'car'},
 {'id': 13, 'name': 'stop sign'},
 {'id': 44, 'name': 'bottle'},
 {'id': 47, 'name': 'cup'},
 {'id': 48, 'name': 'fork'},
 {'id': 49, 'name': 'knife'},
 {'id': 51, 'name': 'bowl'},
 {'id': 62, 'name': 'chair'},
 {'id': 64, 'name': 'potted plant'},
 {'id': 70, 'name': 'toilet'},
 {'id': 72, 'name': 'tv'},
 {'id': 73, 'name': 'laptop'},
 {'id': 74, 'name': 'mouse'},
 {'id': 76, 'name': 'keyboard'},
 {'id': 78, 'name': 'microwave'},
 {'id': 79, 'name': 'oven'},
 {'id': 81, 'name': 'sink'},
 {'id': 85, 'name': 'clock'}]

In [10]:
all_categories = set(map(lambda x: x['id'], train_metadata['categories']))

In [48]:
all_categories

{3, 13, 44, 47, 48, 49, 51, 62, 64, 70, 72, 73, 74, 76, 78, 79, 81, 85}

In [9]:
import requests
import json

In [14]:
val_json_path = '../../../CRT code/COCOstuff/annotations/val.json'

In [13]:
os.path.abspath('../../../CRT code/COCOstuff/annotations/val.json')

'/Users/nguyentuan/Mirror/STUDY_DOCUMENTS/FYP/CRT code/COCOstuff/annotations/val.json'

In [17]:
with open(val_json_path, 'rb') as file:
    val_metadata = json.load(file)

In [19]:
val_metadata['images'][0]

{'license': 4,
 'file_name': '000000397133.jpg',
 'coco_url': 'http://images.cocodataset.org/val2017/000000397133.jpg',
 'height': 427,
 'width': 640,
 'date_captured': '2013-11-14 17:02:52',
 'flickr_url': 'http://farm7.staticflickr.com/6116/6255196340_da26cf2c9e_z.jpg',
 'id': 397133}

In [28]:
test_img_dir = '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt_test'

In [29]:
from tqdm import tqdm

In [51]:
test_dict = {}
test_dict['images']      = []
test_dict['annotations'] = []
test_dict['categories']  = train_metadata['categories']
test_dict['licenses']    = train_metadata['licenses']

os.makedirs(test_img_dir, exist_ok=True)


for i in tqdm(range(len(val_metadata['images']))):
    
    
    file_name = val_metadata['images'][i]['file_name']
    
    category_id = val_metadata['annotations'][i]['category_id']
    
    if category_id not in all_categories:
#         print('not in categories')
        continue
        
    if os.path.exists(os.path.join(test_img_dir, file_name)):
        test_dict['images'].append(val_metadata['images'][i])
        test_dict['annotations'].append(val_metadata['annotations'][i])
        continue
        
    
    
    test_dict['images'].append(val_metadata['images'][i])
    test_dict['annotations'].append(val_metadata['annotations'][i])
    
    url = val_metadata['images'][i]['coco_url']
    
    res = requests.get(url, stream=True)
    
    res.raise_for_status()


    with open(os.path.join(test_img_dir, file_name), 'wb') as file:
        file.write(res.content)





# for img_metadata in tqdm(val_metadata['images']):
    
#     file_name = img_metadata['file_name']
    
#     if os.path.exists(os.path.join(test_img_dir, file_name)):
#         continue
    
#     os.makedirs(test_img_dir, exist_ok=True)
#     url = img_metadata['coco_url']
    
#     res = requests.get(url, stream=True)
    
#     res.raise_for_status()
    
#     with open(os.path.join(test_img_dir, file_name), 'wb') as file:
#         file.write(res.content)

100%|███████████████████████████████████████| 5000/5000 [25:55<00:00,  3.21it/s]


In [52]:
with open('../datasets/COCO18_dset_for_CRT_training/val_metadata.json', 'w') as file:
    json.dump(test_dict, file)

In [7]:
with open('../datasets/COCO18_dset_for_CRT_training/val_metadata.json', 'rb') as file:
    test_dict = json.load(file)

In [15]:
test_dict['annotations'][0]

{'segmentation': [[248.37,
   15.92,
   248.37,
   15.92,
   249.32,
   15.92,
   249.32,
   15.92,
   249.32,
   14.01,
   249.32,
   10.19,
   249.32,
   9.23,
   254.1,
   4.46,
   256.01,
   4.46,
   273.2,
   6.37,
   281.8,
   19.74,
   280.85,
   21.65,
   280.85,
   25.47,
   280.85,
   30.25,
   280.85,
   38.85,
   278.94,
   44.58,
   277.02,
   48.4,
   275.11,
   49.35,
   269.38,
   51.27,
   266.52,
   51.27,
   262.7,
   51.27,
   258.87,
   49.35,
   256.01,
   47.44,
   251.23,
   42.67,
   248.37,
   35.98,
   245.5,
   30.25,
   246.46,
   26.43,
   246.46,
   23.56]],
 'area': 1340.0489500000006,
 'iscrowd': 0,
 'image_id': 482100,
 'bbox': [245.5, 4.46, 36.3, 46.81],
 'category_id': 64,
 'id': 18782}

In [16]:
test_dict['categories']

[{'id': 3, 'name': 'car'},
 {'id': 13, 'name': 'stop sign'},
 {'id': 44, 'name': 'bottle'},
 {'id': 47, 'name': 'cup'},
 {'id': 48, 'name': 'fork'},
 {'id': 49, 'name': 'knife'},
 {'id': 51, 'name': 'bowl'},
 {'id': 62, 'name': 'chair'},
 {'id': 64, 'name': 'potted plant'},
 {'id': 70, 'name': 'toilet'},
 {'id': 72, 'name': 'tv'},
 {'id': 73, 'name': 'laptop'},
 {'id': 74, 'name': 'mouse'},
 {'id': 76, 'name': 'keyboard'},
 {'id': 78, 'name': 'microwave'},
 {'id': 79, 'name': 'oven'},
 {'id': 81, 'name': 'sink'},
 {'id': 85, 'name': 'clock'}]

In [ ]:
train.py
import os
import warnings
import argparse
import datetime
import pathlib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from tqdm import tqdm, trange

from test import test
from utils.evaluate_uncertainty import evaluate_uncertainty
from core.config import create_config, save_config
from core.dataset import COCODataset, COCODatasetWithID, COCODatasetGeneral
from core.model import Model
from core.metrics import AccuracyLogger


## Initialization
#
    
parser = argparse.ArgumentParser()
parser.add_argument("--config", type=str, help="Path to config file. If additional commandline options are provided, they are used to modify the specifications in the config file.")
parser.add_argument("--outdir", type=str, default="output/{date:%Y-%m-%d_%H%M}".format(date=datetime.datetime.now()), help="Path to output folder (will be created if it does not exist).")
parser.add_argument("--checkpoint", type=str, help="Path to model checkpoint from which to continue training.")
parser.add_argument("--annotations", type=str, help="Path to COCO-style annotations file.")
parser.add_argument("--imagedir", type=str, help="Path to images folder w.r.t. which filenames are specified in the annotations.")

parser.add_argument("--test_annotations", type=str, help="Path to COCO-style annotations file for model evaluation.")
parser.add_argument("--test_imagedir", type=str, help="Path to images folder w.r.t. which filenames are specified in the annotations for model evaluation.")
parser.add_argument("--test_frequency", type=int, default=1, help="Evaluate model on test data every __ epochs.")

parser.add_argument("--epochs", type=int, default=1, help="Number of epochs to train.")
parser.add_argument("--save_frequency", type=int, default=1, help="Save model checkpoint every __ epochs.")
parser.add_argument("--print_batch_metrics", action='store_true', default=False, help="Set to print metrics for every batch.")

parser.add_argument("--batch_size", type=int, help="Batchsize to use for training.")
parser.add_argument("--learning_rate", type=float, help="Learning rate to use for training.")
parser.add_argument("--imbalance_reweighting", action='store_true', help="Reweight samples in proportion to the number of samples per class.")
parser.add_argument("--num_decoder_heads", type=int, help="Number of decoder heads.")
parser.add_argument("--num_decoder_layers", type=int, help="Number of decoder layers.")
parser.add_argument("--uncertainty_gate_type", type=str, help="Uncertainty gating mechanism to use. Can be one of: 'entropy', 'relative_softmax_distance', 'learned', 'learned_metric'.")
parser.add_argument("--uncertainty_threshold", type=float, help="Uncertainty threshold for the uncertainty gating module. Note that training does not depend on the threshold, the model can still be used with different thresholds later.")
parser.add_argument("--weighted_prediction", action='store_true', default=None, help="If enabled, the model returns an uncertainty-weighted prediction if the uncertainty_gate prediction exceeds the uncertainty threshold.")
args = parser.parse_args()

# Create output directory
pathlib.Path(args.outdir).mkdir(exist_ok=True, parents=True)

# Load config or create a new one
cfg = create_config(args)

dataset = COCODatasetGeneral(cfg.annotations, cfg.imagedir, image_size =(224,224), normalize_means=[0.485, 0.456, 0.406], normalize_stds=[0.229, 0.224, 0.225])
dataloader = DataLoader(dataset, batch_size=cfg.batch_size, num_workers=4, shuffle=True, pin_memory=True, drop_last=True)

NUM_CLASSES = dataset.NUM_CLASSES
cfg.num_classes = NUM_CLASSES
save_config(cfg, args.outdir)
print(cfg)

model = Model.from_config(cfg)

assert(model.TARGET_IMAGE_SIZE == model.CONTEXT_IMAGE_SIZE == dataset.image_size), "Image size from the dataset is not compatible with the encoder."

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)

if cfg.imbalance_reweighting:
    class_weights = torch.true_divide(dataset.relative_annotation_counts.max(), dataset.relative_annotation_counts)
    criterion = nn.CrossEntropyLoss(weight= class_weights.to(device))
else:
    criterion = nn.CrossEntropyLoss()

if cfg.checkpoint is not None:
    print("Initializing from checkpoint {}".format(cfg.checkpoint))
    checkpoint = torch.load(cfg.checkpoint, map_location="cpu")
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
else:
    print("No checkpoint was passed.")
    model.to(device)
    start_epoch = 1

# Tensorboard
writer = SummaryWriter(log_dir=os.path.join(args.outdir, "runs/{date:%Y-%m-%d_%H%M}".format(date=datetime.datetime.now())))
context_images, target_images, bbox, labels = iter(dataloader).next()
writer.add_images("context_image_batch", context_images) # add example context image batch to tensorboard log
writer.add_images("target_image_batch", target_images) # add example target image batch to tensorboard log
with warnings.catch_warnings(): # add_graph method is known to issue a warning
    warnings.simplefilter("ignore")
    writer.add_graph(model, input_to_model=[context_images.to(device), target_images.to(device), bbox.to(device)]) # add model graph to tensorboard log

accuracy_logger_main_branch = AccuracyLogger(dataset.idx2label)
# accuracy_logger_uncertainty_branch = AccuracyLogger(dataset.idx2label)


## Training
#

for epoch in tqdm(range(start_epoch, args.epochs + 1), position=0, desc="Epochs", leave=True):

    model.train() # set train mode
    accuracy_logger_main_branch.reset() # reset accuracy logger every epoch
    # accuracy_logger_uncertainty_branch.reset()

    for i, (context_images, target_images, bbox, labels_cpu) in enumerate(tqdm(dataloader, position=1, desc="Batches", leave=True)):
        context_images = context_images.to(device)
        target_images = target_images.to(device)
        bbox = bbox.to(device)
        labels = labels_cpu.to(device) # keep a copy of labels on cpu to avoid unnecessary transfer back to cpu later

        # output_uncertainty_branch , output_main_branch, output_weighted, uncertainty = model(context_images, target_images, bbox)
        output_main_branch = model(context_images, target_images, bbox)

        # backpropagation through both branches
        optimizer.zero_grad(set_to_none=True)

        # if cfg.uncertainty_gate_type == "learned" or cfg.uncertainty_gate_type == "learned_metric":
        #     loss_uncertainty_estimator = criterion(output_weighted, labels)
        #     loss_uncertainty_estimator.backward(retain_graph=True)    

        # loss_uncertainty_branch = criterion(output_uncertainty_branch, labels)
        # loss_uncertainty_branch.backward(retain_graph=True)

        loss_main_branch = criterion(output_main_branch, labels)
        loss_main_branch.backward()

        optimizer.step()
        
        # log metrics
        # _, predictions_uncertainty_branch = torch.max(output_uncertainty_branch.detach().to("cpu"), 1) # choose idx with maximum score as prediction
        # batch_accuracy_uncertainty_branch = sum(predictions_uncertainty_branch == labels_cpu) / cfg.batch_size
        # batch_loss_uncertainty_branch = loss_uncertainty_branch.item()
        # writer.add_scalar("Batch Accuracy Uncertainty Branch/train", batch_accuracy_uncertainty_branch, i + (epoch - 1) * len(dataloader))
        # writer.add_scalar("Batch Loss Uncertainty Branch/train", batch_loss_uncertainty_branch, i + (epoch - 1) * len(dataloader))
        # accuracy_logger_uncertainty_branch.update(predictions_uncertainty_branch, labels_cpu)

        _, predictions_main_branch = torch.max(output_main_branch.detach().to("cpu"), 1) # choose idx with maximum score as prediction
        batch_accuracy_main_branch = sum(predictions_main_branch == labels_cpu) / cfg.batch_size
        batch_loss_main_branch = loss_main_branch.item()
        writer.add_scalar("Batch Accuracy Main Branch/train", batch_accuracy_main_branch, i + (epoch - 1) * len(dataloader))
        writer.add_scalar("Batch Loss Main Branch/train", batch_loss_main_branch, i + (epoch - 1) * len(dataloader))
        accuracy_logger_main_branch.update(predictions_main_branch, labels_cpu)

        # writer.add_scalar("Batch Uncertainty/train", torch.mean(uncertainty), i + (epoch - 1) * len(dataloader))

        if args.print_batch_metrics:
            print("\t Epoch {}, Batch {}: \t Loss: {} \t Accuracy: {}".format(epoch, i, batch_loss_main_branch, batch_accuracy_main_branch))


    # log metrics
    writer.add_scalar("Total Accuracy Main Branch/train", accuracy_logger_main_branch.accuracy(), epoch * len(dataloader))
    # writer.add_scalar("Total Accuracy Uncertainty Branch/train", accuracy_logger_uncertainty_branch.accuracy(), epoch * len(dataloader))

    print("\nEpoch {}, Train Accuracy: {}".format(epoch, accuracy_logger_main_branch.accuracy()))
    print("{0:20} {1:10}".format("Class", "Accuracy")) # header
    for name, acc in accuracy_logger_main_branch.named_class_accuarcies().items():
        writer.add_scalar("Class Accuracies Main Branch/train/{}".format(name), acc, epoch * len(dataloader))
        print("{0:20} {1:10.4f}".format(name, acc))

    # for name, acc in accuracy_logger_uncertainty_branch.named_class_accuarcies().items():
    #     writer.add_scalar("Class Accuracies Uncertainty Branch/train/{}".format(name), acc, epoch * len(dataloader))

    # save checkpoint and training accuracies
    if epoch % args.save_frequency == 0:
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()}, args.outdir + "/checkpoint_{}.tar".format(epoch))
        print("Checkpoint saved.")

        accuracy_logger_main_branch.save(args.outdir, name="train_accuracies_epoch_{}".format(epoch))
        # accuracy_logger_uncertainty_branch.save(args.outdir, name="train_accuracies_uncertainty_branch_epoch_{}".format(epoch))
    
    # evaluation on test data
    if cfg.test_annotations is not None and cfg.test_imagedir is not None and epoch % args.test_frequency == 0:
        print("Starting evaluation on test data.")
        test_accuracy = test(model, cfg.test_annotations, cfg.test_imagedir, outdir=args.outdir, epoch=epoch)

        writer.add_scalar("Total Accuracy/test", test_accuracy.accuracy(), epoch * len(dataloader))
        for name, acc in test_accuracy.named_class_accuarcies().items():
            writer.add_scalar("Class Accuracies/test/{}".format(name), acc, epoch * len(dataloader))

        # print("Starting uncertainty evaluation.")
        # test_uncertainty_log = evaluate_uncertainty(model, cfg.test_annotations, cfg.test_imagedir)
        # writer.add_figure("Uncertainty Threshold Curve", test_uncertainty_log.plot_accuracy_vs_threshold(), epoch * len(dataloader))

        # if (args.epochs - epoch) / args.test_frequency < 1: # last evaluation
        #     writer.add_hparams({"learning_rate": cfg.learning_rate, "num_decoder_layers": cfg.num_decoder_layers, "num_decoder_heads": cfg.num_decoder_heads,
        #                         "uncertainty_gate_type": cfg.uncertainty_gate_type, "uncertainty_threshold": cfg.uncertainty_threshold, "imbalance_reweighting": str(cfg.imbalance_reweighting)},
        #                         metric_dict={"hparam/accuracy": test_accuracy.accuracy()})
        
writer.close()